# GRU para formación de colas (CAETEC)

Entrena y evalúa `GRUModel` con `TimeSeriesSplit` de 5 folds, muestra métricas por fold y las compara contra líneas base.

`GRUModel` hereda de `TCNModel` la preparación de ventanas por robot (ventana en *t−1*, etiqueta en *t*), el escalado ajustado solo con train y la evaluación; solo cambia la arquitectura. Así los resultados son comparables con el TCN.

## 1. Configuración

In [ ]:
# ====== RUTAS (ajusta a tu Drive) ======
DATA_PATH = "/content/drive/MyDrive/Datos/Procesado/dataset_final_serie_tiempo.parquet"
MODELING_PARENT = "/content/modeling"

# ====== EXPERIMENTO-00 ======
TASK = "regression" # "regression" (pct_posible_cola) | "classification" (hay_cola)
WINDOW = 48 # 12 (6 h) | 24 (12 h) | 48 (24 h)
VERSION = "base" # "base" | "norm" | "bi"
LEARNING_RATE = 1e-3
EPOCHS = 50
BATCH = 128
N_SPLITS = 5
QUICK = False # False = Entrenamiento completo | True = prueba rápida (1 fold, 3 épocas)
TUNE_THRESHOLD = True # Busca el umbral con mejor F1 en la validación de cada fold (nunca en test)
SEED = 42
TARGET = "pct_posible_cola"

## 2. Imports y datos

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
# ---- Encontrar el paquete `modeling` (gru_model.py, tcn_model.py, model_base.py) ----
# 1) Si ya están en MODELING_PARENT/modeling/ se usan esos.
# 2) Si no, se clona el repo (rama REPO_BRANCH) y se usa vms/notebooks/modeling/.
# 3) Si gru_model.py aún no está en GitHub, se piden los 3 .py para subirlos a mano.
REPO_URL = "https://github.com/kamimartinez/VMS-prediction.git"
REPO_BRANCH = "main"
NEEDED = ["gru_model.py", "tcn_model.py", "model_base.py"]

def _has_files(folder):
    return all(os.path.exists(os.path.join(folder, "modeling", f)) for f in NEEDED)

if not _has_files(MODELING_PARENT) and IN_COLAB:
    os.system(f"rm -rf /content/vms && git clone -q -b {REPO_BRANCH} {REPO_URL} /content/vms")
    if _has_files("/content/vms/notebooks"):
        MODELING_PARENT = "/content/vms/notebooks"
    else:
        print("gru_model.py no está en GitHub todavía. Sube gru_model.py, tcn_model.py y model_base.py:")
        from google.colab import files
        os.makedirs(os.path.join(MODELING_PARENT, "modeling"), exist_ok=True)
        for name, data in files.upload().items():
            with open(os.path.join(MODELING_PARENT, "modeling", name), "wb") as fh:
                fh.write(data)

if not _has_files(MODELING_PARENT):
    raise FileNotFoundError(f"Faltan {NEEDED} en {MODELING_PARENT}/modeling/")
if MODELING_PARENT not in sys.path:
    sys.path.insert(0, MODELING_PARENT)
print("Usando módulos de:", os.path.join(MODELING_PARENT, "modeling"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, mean_absolute_error, mean_squared_error, r2_score)
from tensorflow.keras.callbacks import EarlyStopping

from modeling.gru_model import GRUModel

np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
df = pd.read_parquet(DATA_PATH)
# prepare() espera temporada como texto tipo object (con pandas 3 llega como 'str')
df["temporada"] = df["temporada"].astype(object)
print(df.shape)
print(df.dtypes.to_string())
df.head()

## 3. Ventanas 3D por robot

La ventana termina en el bloque *t−1* y la etiqueta es la del bloque *t*, así no se usan las variables mecánicas del bloque que se quiere predecir.

In [ ]:
gru = GRUModel(task=TASK, window=WINDOW, learning_rate=LEARNING_RATE)
X_seq, y_seq, meta = gru.prepare(df, df[TARGET])
print("X_seq:", X_seq.shape, "(N, ventana, features)")
print("y_seq:", y_seq.shape)
print("Features:", gru.feature_names_)
print("Periodo objetivo:", meta["bloque_tiempo"].min(), "->", meta["bloque_tiempo"].max())
print("Ventanas por robot:\n", meta["ms"].value_counts().to_string())
if TASK == "classification":
    print(f"Proporción hay_cola=1: {y_seq.mean():.3f}")
else:
    print(f"pct_posible_cola promedio: {y_seq.mean():.3f}")

## 4. Líneas base

- **Mayoritaria / media**: predice siempre lo más frecuente del train.
- **Persistencia**: repite el valor del bloque anterior del mismo robot (dato disponible al predecir).

In [ ]:
# Valor del bloque anterior (t-1) del mismo robot, alineado con cada ventana
lookup = df[["ms", "bloque_tiempo", TARGET]].copy()
lookup["bloque_tiempo"] = pd.to_datetime(lookup["bloque_tiempo"]) + pd.Timedelta(minutes=30)
lookup = lookup.rename(columns={TARGET: "prev"})
prev = meta[["ms", "bloque_tiempo"]].merge(lookup, on=["ms", "bloque_tiempo"], how="left")["prev"].to_numpy()
persist = np.nan_to_num(prev, nan=np.nanmean(prev))
persist_bin = (persist > gru.threshold).astype(float)
print("Persistencia disponible en", (~np.isnan(prev)).mean().round(3), "de las ventanas")

## 5. Funciones de métricas

In [ ]:
def cls_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true).astype(int); y_pred = (np.asarray(y_prob) >= thr).astype(int)
    try: auc = roc_auc_score(y_true, y_prob)
    except ValueError: auc = np.nan
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "AUC": auc}

def reg_metrics(y_true, y_pred, thr=0.5):
    # Además de MAE/RMSE/R2 se reporta F1 de hay_cola (> 0.5) para comparar con el criterio de éxito (F1 > 0.60)
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    cola_true = (y_true > gru.threshold).astype(int); cola_pred = (y_pred >= thr).astype(int)
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "R2": r2_score(y_true, y_pred),
            "F1_cola": f1_score(cola_true, cola_pred, zero_division=0)}

def metrics(y_true, y_hat, thr=0.5):
    return cls_metrics(y_true, y_hat, thr) if TASK == "classification" else reg_metrics(y_true, y_hat, thr)

## 6. Entrenamiento con TimeSeriesSplit

In [ ]:
epochs = 3 if QUICK else EPOCHS
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
folds = list(tscv.split(X_seq))
if QUICK:
    folds = folds[:1]
    print("MODO RÁPIDO: 1 fold, 3 épocas")

rows_model, rows_active, rows_base, rows_persist = [], [], [], []
histories, thresholds, last = [], [], {}

for k, (tr, te) in enumerate(folds, start=1):
    print(f"\n===== Fold {k}/{len(folds)} | train={len(tr):,} test={len(te):,} =====")
    print("Test:", meta["bloque_tiempo"].iloc[te].min(), "->", meta["bloque_tiempo"].iloc[te].max())
    tf.keras.backend.clear_session()

    # Validación cronológica: el último 10 % del train (no aleatorio)
    n_val = max(1, int(len(tr) * 0.1))
    tr_fit, va = tr[:-n_val], tr[-n_val:]

    X_tr, X_va, X_te = gru.scale(X_seq[tr_fit], X_seq[va], X_seq[te])   # scaler ajustado solo con train
    y_tr, y_va, y_te = y_seq[tr_fit], y_seq[va], y_seq[te]

    model = gru.compile(input_shape=X_tr.shape[1:], v=VERSION)
    if k == 1:
        model.summary()

    hist = model.fit(X_tr, y_tr, epochs=epochs, batch_size=BATCH, validation_data=(X_va, y_va),
                     class_weight=gru.class_weights(y_tr),          # None en regresión
                     callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
                     verbose=2)
    histories.append(hist.history)

    thr = gru.best_threshold(model, X_va, y_va)[0] if TUNE_THRESHOLD else 0.5
    thresholds.append(thr)
    print(f"Umbral usado en el fold {k}: {thr:.2f}")

    y_hat = model.predict(X_te, batch_size=512, verbose=0).ravel()
    rows_model.append({"fold": k, "umbral": thr, **metrics(y_te, y_hat, thr)})

    # Solo bloques con ordeños (análisis retrospectivo)
    if "hubo_ordenios" in meta.columns:
        act = meta["hubo_ordenios"].to_numpy()[te] == 1
        rows_active.append({"fold": k, **metrics(y_te[act], y_hat[act], thr)})

    # Líneas base
    if TASK == "classification":
        majority = float(y_tr.mean() > 0.5)
        rows_base.append({"fold": k, **cls_metrics(y_te, np.full(len(y_te), majority))})
        rows_persist.append({"fold": k, **cls_metrics(y_te, persist_bin[te])})
    else:
        rows_base.append({"fold": k, **reg_metrics(y_te, np.full(len(y_te), y_tr.mean()))})
        rows_persist.append({"fold": k, **reg_metrics(y_te, persist[te])})

    last = dict(model=model, history=hist, X_te=X_te, y_te=y_te, y_hat=y_hat, te=te, thr=thr)

## 7. Resultados por fold

In [ ]:
def summarize(rows, name):
    d = pd.DataFrame(rows).set_index("fold")
    d.loc["media"] = d.mean(); d.loc["std"] = d.iloc[:-1].std()
    print(f"\n--- {name} ---"); display(d.round(4)); return d

res_model   = summarize(rows_model,   f"GRU ({VERSION}, ventana={WINDOW}) - todos los bloques")
if rows_active:
    res_active = summarize(rows_active, "GRU - solo bloques con ordeños")
res_base    = summarize(rows_base,    "Base: " + ("clase mayoritaria" if TASK == "classification" else "media del train"))
res_persist = summarize(rows_persist, "Base: persistencia (bloque anterior)")

## 8. Comparación contra líneas base

El porcentaje de bloques con cola cambia mucho entre folds (sube con el tiempo), así que la exactitud sola engaña. Mira F1 (criterio de éxito: **F1 > 0.60**; en regresión se usa `F1_cola`) y que la GRU supere a la persistencia.

In [ ]:
key = "F1" if TASK == "classification" else "F1_cola"
n = len(folds); x = np.arange(1, n + 1); w = 0.27

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - w, res_model[key].iloc[:n],   w, label="GRU")
ax.bar(x,     res_persist[key].iloc[:n], w, label="Persistencia")
ax.bar(x + w, res_base[key].iloc[:n],    w, label="Mayoritaria/media")
ax.axhline(0.60, color="red", ls="--", label="Criterio F1 = 0.60")
ax.set_xticks(x); ax.set_xlabel("Fold"); ax.set_ylabel(key)
ax.set_title(f"{key} por fold"); ax.legend(); plt.tight_layout(); plt.show()

print(f"{key} medio  GRU: {res_model.loc['media', key]:.4f} | "
      f"persistencia: {res_persist.loc['media', key]:.4f} | "
      f"base: {res_base.loc['media', key]:.4f}")
if TASK == "regression":
    print(f"MAE medio  GRU: {res_model.loc['media', 'MAE']:.4f} | "
          f"persistencia: {res_persist.loc['media', 'MAE']:.4f} | "
          f"base: {res_base.loc['media', 'MAE']:.4f}")

## 9. Curvas de entrenamiento (todos los folds)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for k, h in enumerate(histories, start=1):
    axes[0].plot(h["loss"], label=f"fold {k}")
    axes[1].plot(h["val_loss"], label=f"fold {k}")
axes[0].set_title("Loss entrenamiento"); axes[1].set_title("Loss validación")
for a in axes: a.set_xlabel("Epoch"); a.legend()
plt.tight_layout(); plt.show()

## 10. Detalle del último fold (curvas, matriz de confusión / ROC o residuales)

In [ ]:
df_pred, df_metrics = gru.evaluate(last["model"], last["history"], last["X_te"], last["y_te"], threshold=last["thr"])

## 11. Predicción vs. real en una semana (un robot)

In [ ]:
m = meta.iloc[last["te"]].copy()
m["real"] = last["y_te"]; m["pred"] = last["y_hat"]
robot = m["ms"].iloc[0]
s = m[m["ms"] == robot].head(336)          # 336 bloques = 7 días

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s["bloque_tiempo"], s["real"], label="Real", alpha=0.7)
ax.plot(s["bloque_tiempo"], s["pred"], label="Predicho (prob./valor)", alpha=0.9)
ax.set_title(f"{robot}: primera semana del último fold"); ax.legend()
plt.tight_layout(); plt.show()

## 12. Importancia de variables (por permutación, último fold)

Desordena una variable a la vez y mide cuánto empeora la métrica. Más caída = más importante. Se muestrea para que corra rápido.

In [ ]:
rng = np.random.default_rng(SEED)
Xp, yp = last["X_te"], last["y_te"]
idx = rng.choice(len(Xp), size=min(5000, len(Xp)), replace=False)
Xs, ys = Xp[idx], yp[idx]
mkey = "F1" if TASK == "classification" else "MAE"

base_val = metrics(ys, last["model"].predict(Xs, batch_size=512, verbose=0).ravel(), last["thr"])[mkey]
imp = {}
for j, name in enumerate(gru.feature_names_):
    Xperm = Xs.copy()
    Xperm[:, :, j] = Xperm[rng.permutation(len(Xs)), :, j]    # permuta la variable entre muestras
    v = metrics(ys, last["model"].predict(Xperm, batch_size=512, verbose=0).ravel(), last["thr"])[mkey]
    imp[name] = (base_val - v) if TASK == "classification" else (v - base_val)

imp = pd.Series(imp).sort_values()
imp.plot.barh(figsize=(8, 7)); plt.title(f"Importancia por permutación (caída/aumento en {mkey})")
plt.tight_layout(); plt.show()

## 13. Guardar resultados

In [ ]:
out = res_model.copy(); out.insert(0, "config", f"{TASK}|{VERSION}|w{WINDOW}|lr{LEARNING_RATE}")
out.to_csv(f"resultados_gru_{TASK}_{VERSION}_w{WINDOW}.csv")
print("Guardado en el directorio actual de Colab (cópialo a Drive si lo quieres conservar).")

## 14. Exportar Modelo

In [ ]:
import json, joblib
from pathlib import Path

MODELS_DIR = Path(MODELING_PARENT) / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

tag = f"gru_{TASK}_{VERSION}_w{WINDOW}" + ("_quick" if QUICK else "")
model_file  = MODELS_DIR / f"{tag}.keras"
scaler_file = MODELS_DIR / f"{tag}_scaler.joblib"
config_file = MODELS_DIR / f"{tag}_config.json"

last["model"].save(model_file)
joblib.dump(gru.scaler, scaler_file)
json.dump({
    "task": TASK, "window": WINDOW, "version": VERSION, "learning_rate": LEARNING_RATE,
    "threshold_label": gru.threshold, "threshold_decision": last["thr"], "target": TARGET,
    "feature_names": gru.feature_names_,
    "input_shape": list(last["X_te"].shape[1:]),
    "shift_target": True,
    "metrics_mean": res_model.loc["media"].to_dict(),
}, open(config_file, "w"), indent=2, ensure_ascii=False)

print("Guardado:")
for p in (model_file, scaler_file, config_file):
    print(" ", p)

## 15. Verificar exportación

In [ ]:
loaded = tf.keras.models.load_model(model_file)
p_orig   = last["model"].predict(last["X_te"][:1000], verbose=0).ravel()
p_loaded = loaded.predict(last["X_te"][:1000], verbose=0).ravel()
print("Máx. diferencia:", float(np.abs(p_orig - p_loaded).max()))   # debe ser ~0